## 3.4 MLP基础ANN - 参数初始化方法

#### 1. 常见初始化方法总览
先建立整体印象：
* 普通随机初始化
* Xavier 初始化（Glorot Initialization）
* He 初始化（Kaiming Initialization）

可以先把它们理解成三代思路：
* 第一代：随便给一点小随机数
* 第二代：考虑输入输出维度，控制方差，得到 Xavier
* 第三代：进一步考虑 ReLU 的特点，得到 He 初始化

#### 2. 普通随机初始化（Random Initialization）

##### 2.1 最基础的做法
常见的两种普通随机方式：
* 均匀分布初始化 (Uniform Initialization)
    * 权重从一个指定的区间 [a,b] 中等概率抽取。
    * 在早期的框架中，常使用 [−0.05,0.05] 或 [−1,1] 这种固定的小区间。
* 正态（高斯）分布初始化 (Normal/Gaussian Initialization)
    * 权重服从均值为 0、标准差为 σ（通常是一个很小的数，如 0.01）的正态分布。

##### 2.2 这种方法的问题
虽然它比“全 0 初始化”好很多，但它仍然比较粗糙，因为它没有考虑：
* 输入维度是多少
* 输出维度是多少
* 当前层用的是什么激活函数

所以在浅层网络里也许还能用，但在更深的网络中不够稳定。

#### 3. Xavier 初始化（Glorot Initialization）

##### 3.1 Xavier 初始化想解决什么问题？
它的目标是：
```
想象信号（数据）流经一个深层神经网络。
如果每一层的权重让信号缩小了一点，经过 50 层后，信号就几乎变成 0 了（梯度消失）；
反之，如果每一层都放大了一点，信号就会爆炸（梯度爆炸）。
Xavier 初始化的目标： 保持信号在传递过程中的稳定性，确保信息能在大规模网络中顺畅流动
```
通俗理解：
* 不希望前一层输出到下一层后突然特别大
* 也不希望传着传着越来越小

希望每层的数值规模都比较“平稳”。

##### 3.2 Xavier 初始化适合什么激活函数？
Xavier 初始化更适合这些激活函数：
* Sigmoid
* Tanh
* 或其他比较对称、接近线性的激活函数
因为它的理论推导更符合这类函数的特性。

##### 3.3 Xavier 初始化的核心思想
它会根据：
* 输入神经元个数 fan_in
* 输出神经元个数 fan_out
来自动决定初始化范围，而不是手动瞎设。

常见形式有两种：
1. Xavier Normal
    * 权重 W 服从均值为 0，方差为 σ2 的正态分布：
    * `σ = sqrt(2/(fan_in + fan_out))`
    * `[0, sqrt(2/(fan_in + fan_out))]`
2. Xavier Uniform
    * 权重 W 从以下均匀分布区间内随机采样：
    * `[-sqrt(6/(fan_in + fan_out)), sqrt(6/(fan_in + fan_out))]`

Xavier 会根据层的输入输出规模，自动调整初始化范围。

#### 4. He 初始化（Kaiming Initialization）

##### 4.1 为什么还需要 He 初始化？
因为后来人们大量使用：
* ReLU
* Leaky ReLU

这类激活函数。

而 Xavier 并不是专门为 ReLU 设计的。

由于 ReLU 会把一部分负值直接变成 0，因此它会改变激活值的分布。

所以需要一种更适合 ReLU 的初始化方法，这就是：He Initialization

##### 4.2 He 初始化适合什么场景？
主要适合：
* ReLU
* Leaky ReLU
* 以及 ReLU 家族激活函数

这也是为什么现代神经网络中，He 初始化特别常见。

##### 4.3 He 初始化的核心思想
它主要根据 fan_in 来控制方差。

常见形式：
1. He Normal
    * 权重 W 服从均值为 0，方差为 σ2 的正态分布：
    * `σ = sqrt(2/fan_in)`
    * `[0, sqrt(2/fan_in)]`
2. He Uniform
    * 权重 W 从以下均匀分布区间内随机采样：
    * `[-sqrt(6/fan_in), sqrt(6/fan_in)]`

He 初始化是专门为了适配 ReLU 这类激活函数，让训练更稳定。

#### 5. 偏置一般怎么初始化？
偏置通常比权重简单得多。

常见做法：
* 初始化为 0
* 或初始化为很小的常数

大多数情况下：

✅ bias 设为 0 是完全常见且合理的

因为偏置不会像权重那样引起对称性问题的核心矛盾。

#### 6. PyTorch 中的初始化方式
在 PyTorch 中，很多层已经有默认初始化方式，例如：
* nn.Linear
* nn.Conv2d

都会自动初始化参数。

但在很多情况下，我们也可以手动指定初始化方式。

##### 6.1 查看参数

In [ ]:
import torch 
import torch.nn as nn

linear = nn.Linear(3, 4)

# 这里会看到系统已经自动初始化好的权重和偏置。
print(linear.weight)
print(linear.bias)

Parameter containing:
tensor([[-0.1591,  0.4257, -0.1290],
        [ 0.5056, -0.1138, -0.0417],
        [-0.0459, -0.4771,  0.4943],
        [ 0.3458,  0.0831, -0.2297]], requires_grad=True)
Parameter containing:
tensor([-0.2084,  0.2198,  0.0688, -0.1920], requires_grad=True)


##### 6.2 手动初始化 Xavier

In [2]:
import torch.nn.init as init

linear_2 = nn.Linear(3,4)

init.xavier_uniform_(linear_2.weight)
init.zeros_(linear_2.bias)

print(linear_2.weight)
print(linear_2.bias)

Parameter containing:
tensor([[-0.2627, -0.5858, -0.4160],
        [ 0.4966, -0.3186, -0.7994],
        [ 0.0188, -0.2790, -0.2938],
        [ 0.8147, -0.3001,  0.6002]], requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)


##### 6.3 手动初始化 He

In [4]:
linear_3 = nn.Linear(3,4)

init.kaiming_uniform_(linear_3.weight, nonlinearity='relu')
init.zeros_(linear_3.bias)
print(linear_3.weight)
print(linear_3.bias)

Parameter containing:
tensor([[ 0.1375,  0.4900,  0.1656],
        [ 0.9949, -0.5706, -0.0527],
        [ 1.3895, -1.3479, -0.7153],
        [ 0.3226, -0.3307, -1.0409]], requires_grad=True)
Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)


#### 7. 参数初始化与前面学的知识如何连接
现在可以把前面的知识串起来了：

##### 7.1 我们已经知道一层神经网络在做什么
每层本质上是：
* 线性变换：Z = XW^T + b
* 激活函数：A = activation(Z)

##### 7.2 现在我们知道 W 和 b 从哪里来
它们不是凭空出现的，而是在训练开始前：
* 先初始化
* 再通过训练不断更新

也就是说：**初始化是参数学习的起点。**

##### 7.3 初始化影响整个训练闭环
初始化不好，会导致：
* forward 数值不稳定
* activation 分布不合理
* backward 梯度异常
* optimizer 更新困难

所以初始化虽然只是第一步，但它会影响后面所有步骤。